# Test CLUE performance

In this notebook, we test CLUE's performance for 3 different setups of a Bayesian last-layer MNIST classifier. 

We allow a choice between:
- Classifier dominated backbone 
- Joint training backbone
- Autoencoder dominated backbone

## Setup


In [1]:
inDrive = False

In [2]:
import sys
import os

if inDrive:
    from google.colab import drive
    drive.mount('/content/drive')
    os.chdir('/content/drive/My Drive/Hybrid-CLUE/MyImplementation/testing_notebooks')

# Get current directory and parent directory
current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)

# Add parent directory to path if not already there
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

Import libraries

In [3]:
import importlib
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
import os

import models.BLL
import models.mnist_classifier_BLL
import train
import sampler
import models.regene_models


Set the configuration

In [4]:
# 1. Configuration
class Config:
    # Data
    batch_size = 512
    latent_dim = 256

    # Hardware
    device = 'cuda' if torch.cuda.is_available() \
    else 'mps' if torch.backends.mps.is_available() \
    else 'cpu'

cfg = Config()

In [5]:
models_dir = '../model_saves/new_regene_models'
results_dir = '../results/new_regene_models'
os.makedirs(models_dir, exist_ok=True)
os.makedirs(results_dir, exist_ok=True)

## Backbone
We'll start with the backbone loading. Options are either classifier_dominated, joint_training, or autoencoder_dominated.

In [6]:
classifier_dominated = False
joint_training = True
autoencoder_dominated = False

### Load chosen model

Load model

In [ ]:
importlib.reload(models.regene_models)

decoder = models.regene_models.Decoder(latent_dim=cfg.latent_dim, device=cfg.device)
backbone = models.regene_models.Classifier(latent_dim=cfg.latent_dim, num_classes=10, device=cfg.device)

if classifier_dominated:
    backbone.load(models_dir + '/classifier_dominated_classifier_256.pt')
    decoder.load(models_dir + '/classifier_dominated_decoder_256.pt')
    name = 'classifier_dominated'
    bll_name = 'BLL_VI_classifier_dominated_first_256.pt'
elif joint_training:
    backbone.load(os.path.join(models_dir, 'joint_classifier_256.pt'))
    decoder.load(os.path.join(models_dir, 'joint_decoder_256.pt'))
    name = 'joint_training'
    bll_name = 'BLL_VI_joint_training_first_256.pt'
elif autoencoder_dominated:
    backbone.load(models_dir + '/autoencoder_dominated_classifier_full_256.pt')
    decoder.load(models_dir + '/autoencoder_dominated_decoder_256.pt')
    name = 'autoencoder_dominated'
    bll_name = 'BLL_VI_autoencoder_dominated_first_256.pt'

Load the Datasets

In [8]:
# Load the MNIST dataset
transform = transforms.Compose([transforms.ToTensor()])
trainset = torchvision.datasets.MNIST(root='../data', train=True, download=True, transform=transform)
testset = torchvision.datasets.MNIST(root='../data', train=False, download=True, transform=transform)

# Split training set into train and validation
train_size = int(0.8 * len(trainset))
val_size = len(trainset) - train_size
trainset, valset = torch.utils.data.random_split(trainset, [train_size, val_size])

trainloader = torch.utils.data.DataLoader(trainset, batch_size=cfg.batch_size, shuffle=True, num_workers=2)
valloader = torch.utils.data.DataLoader(valset, batch_size=cfg.batch_size, shuffle=False, num_workers=2)
testloader = torch.utils.data.DataLoader(testset, batch_size=cfg.batch_size, shuffle=False, num_workers=2)

Create a models and results directory if it doesn't exist

## Bayesian Last Layer - VI
Next we'll train a variational inference version of the Bayesian last layer using the chosen backbone.


In [9]:
if inDrive: 
    %pip install torchbnn

### Load the model

In [ ]:
from models.BLL_VI import BayesianLastLayerVI
importlib.reload(models.BLL_VI)

import os
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'

bll_vi = BayesianLastLayerVI(
    backbone=backbone,
    input_dim=cfg.latent_dim,  # Matches backbone's encoder output
    output_dim=10,  # MNIST classes
    device=cfg.device
)

# Verify all model components are on the correct device
print(f"Backbone device: {next(bll_vi.backbone.parameters()).device}")
print(f"Last layer device: {next(bll_vi.last_layer.parameters()).device}")

bll_vi.load_checkpoint(os.path.join(models_dir, bll_name))

### Test the model
To check for Bayesian behaviour, we find an uncertain prediction and visualize multiple samples from the posterior.


In [ ]:
import torch.nn.functional as F

# Search for an uncertain prediction and visualize multiple samples
test_iter = iter(testloader)
found_uncertain = False
max_tries = 100
num_samples = 5

try:
    while not found_uncertain:
        x, y = next(test_iter)
        # Look at each example in the batch
        for i in range(len(x)):
            x_single = x[i:i+1]
            y_single = y[i:i+1]

            # Get multiple predictions for this single example
            outputs = []
            with torch.no_grad():  # Add no_grad context
                for _ in range(num_samples):
                    logits = bll_vi(x_single)
                    probs = F.softmax(logits, dim=1)
                    outputs.append(probs)

            # Stack predictions
            probs = torch.stack(outputs)  # Shape: [num_samples, 1, num_classes]
            probs = probs.squeeze(1)  # Remove batch dimension -> [num_samples, num_classes]

            # Check if predictions are not all highly confident
            max_probs = probs.max(dim=1)[0]
            if max_probs.mean() < 0.9:  # If average confidence is less than 60%
                found_uncertain = True
                break

except StopIteration:
    if not found_uncertain:
        print("Could not find uncertain prediction in entire test set")

# Plot the softmax distributions
plt.figure(figsize=(10, 6))
x_axis = range(probs.shape[1])  # Range over number of classes

for i in range(num_samples):
    plt.plot(x_axis, probs[i].detach().cpu().numpy(), 'o-', alpha=0.9, label=f'Sample {i+1}')

plt.xlabel('Class')
plt.ylabel('Probability')
plt.title(f'VI Model: Softmax Distribution Samples for Uncertain Test Example\nTrue Class: {y_single.item()}')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Display the uncertain image
plt.figure(figsize=(5, 5))
img = x_single.squeeze().cpu()
if img.shape[0] == 1:  # If grayscale, remove channel dimension
    img = img.squeeze(0)
plt.imshow(img, cmap='gray' if len(img.shape) == 2 else None)
plt.axis('off')
plt.title(f'Uncertain Image (True Class: {y_single.item()})')
plt.show()

In [ ]:
# Get test images
test_images, test_labels = next(iter(testloader))
test_images = test_images.to(cfg.device)

# Get reconstructions using the classifier and decoder
with torch.no_grad():
    # Get latent representations from classifier
    latent_reps, _ = backbone(test_images)  # Returns (latent_rep, logits)
    # Reconstruct using decoder
    reconstructions = decoder(latent_reps)

# Plot original and reconstructed images
fig, axes = plt.subplots(2, 8, figsize=(20, 5))
fig.suptitle('Original vs Reconstructed Images', fontsize=16)

# Plot original images
for i in range(8):
    axes[0,i].imshow(test_images[i].cpu().squeeze(), cmap='gray')
    axes[0,i].axis('off')
    if i == 0:
        axes[0,i].set_title('Original', pad=10)

# Plot reconstructions
for i in range(8):
    axes[1,i].imshow(reconstructions[i].cpu().squeeze(), cmap='gray')
    axes[1,i].axis('off')
    if i == 0:
        axes[1,i].set_title('Reconstructed', pad=10)

plt.tight_layout()
plt.show()

# Optionally, calculate reconstruction error
mse = nn.MSELoss()
recon_error = mse(reconstructions, test_images)
print(f'Reconstruction MSE: {recon_error.item():.4f}')

### Get Uncertain Images

In [13]:
import importlib
import clue.evaluate_CLUE
importlib.reload(clue.evaluate_CLUE)
from clue.evaluate_CLUE import evaluate_clue_counterfactuals, find_uncertain_images, visualize_counterfactual_results, evaluate_single_clue_counterfactual

In [14]:
uncertain_images, uncertain_indices = find_uncertain_images(bll_vi, testloader, n=50, device=cfg.device)

In [ ]:
# Plot the first 5 uncertain images
import matplotlib.pyplot as plt

plt.figure(figsize=(15, 3))
for i in range(5):
    plt.subplot(1, 5, i+1)
    plt.imshow(uncertain_images[i, 0].cpu(), cmap='gray')
    plt.title(f"Index: {uncertain_indices[i]}")
    plt.axis('off')
plt.tight_layout()
plt.show()

print("Uncertainty indices of these images:", uncertain_indices[:5].tolist())

## Ensemble Last Layer

### Load the model

In [ ]:
from models.ensemble_LL import EnsembleLastLayer

import os
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'

ensemble_model = EnsembleLastLayer(
    backbone=backbone,
    input_dim=cfg.latent_dim,  # Matches backbone's encoder output
    output_dim=10,  # MNIST classes
    n_members=50,
    device=cfg.device
)

# Verify all model components are on the correct device
print(f"Backbone device: {next(ensemble_model.backbone.parameters()).device}")
print(f"Last layer device: {next(ensemble_model.ensemble.parameters()).device}")

In [ ]:
ensemble_model.load_checkpoint(models_dir + '/ensemble_model.pt')

## Visualise gradients

### Full space

In [ ]:
import clue.evaluate_latent_space
from clue.evaluate_latent_space import create_2d_latent_visualization, create_grid_in_latent_space, analyze_multiple_target_classes
importlib.reload(clue.evaluate_latent_space)

In [ ]:
latent_vis = create_2d_latent_visualization(
    dataloader=trainloader, 
    model=bll_vi, 
    device=cfg.device,
    method='pca',
    n_samples=2000
)

In [ ]:
grid_data = create_grid_in_latent_space(
    reduced_latents = latent_vis['reduced_latents'],
    inverse_transform = latent_vis['inverse_transform'],
    grid_size = 200
)

In [ ]:
results = analyze_multiple_target_classes(
    bll_vi, backbone, 
    grid_data, target_classes=[0, 1, 2], 
    output_dir='./visualization_results'
)

### Path between centroids

In [18]:
import clue.evaluate_latent_space
importlib.reload(clue.evaluate_latent_space)
from clue.evaluate_latent_space import compute_class_centroids, analyze_class_to_class_path, analyze_class_to_class_path_probs, analyze_probability_gradients

In [19]:
centroids = compute_class_centroids(bll_vi, testloader, classes=[1, 2, 3, 4, 5, 6, 7, 8, 9])

In [ ]:
results = analyze_class_to_class_path(
    bayes_model=bll_vi, 
    det_model=backbone, 
    centroids=centroids, 
    start_class=3, 
    target_class=6, 
    n_points=200)

In [ ]:
results = analyze_class_to_class_path(
    bayes_model=bll_vi, 
    det_model=backbone, 
    centroids=centroids, 
    start_class=3, 
    target_class=6, 
    n_points=200)

In [ ]:
results = analyze_class_to_class_path_probs(
    bayes_model=bll_vi, 
    det_model=backbone, 
    centroids=centroids, 
    start_class=3, 
    target_class=6, 
    n_points=200)

In [ ]:
results = analyze_probability_gradients(
    bayes_model=bll_vi, 
    det_model=backbone, 
    centroids=centroids, 
    start_class=3, 
    target_class=6, 
    n_points=200,
    num_samples=100)

In [ ]:
# Let's create a more detailed visualization of the probability gradients
def plot_detailed_gradient_analysis(results, start_class, target_class, figsize=(16, 12)):
    """
    Create a more detailed visualization of probability gradients analysis results.
    
    Args:
        results: Dictionary with analysis results from analyze_probability_gradients
        start_class: Starting class index
        target_class: Target class index
        figsize: Figure size
    """
    alphas = results['alphas']
    bayes_probs = results['bayes_probs']
    det_probs = results['det_probs']
    bayes_grad_norms = results['bayes_grad_norms']
    det_grad_norms = results['det_grad_norms']
    
    # Create a 1x2 subplot figure
    fig, axes = plt.subplots(1, 2, figsize=(figsize[0], figsize[1]//2))
    
    # Plot 1: Probabilities
    axes[0].plot(alphas, bayes_probs, 'b-', linewidth=2, label='Bayesian')
    axes[0].plot(alphas, det_probs, 'r--', linewidth=2, label='Deterministic')
    axes[0].set_title(f'Target Class Probability: Class {start_class} → Class {target_class}')
    axes[0].set_xlabel('Interpolation Parameter (α)')
    axes[0].set_ylabel('Probability')
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()
    
    # Plot 2: Gradient norms
    axes[1].plot(alphas, bayes_grad_norms, 'b-', linewidth=2, label='Bayesian')
    axes[1].plot(alphas, det_grad_norms, 'r--', linewidth=2, label='Deterministic')
    axes[1].set_title(f'Gradient of Raw Probability (||∇P(class={target_class})||)')
    axes[1].set_xlabel('Interpolation Parameter (α)')
    axes[1].set_ylabel('Gradient Norm')
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()
    
    plt.tight_layout()
    plt.show()
    
    # Print correlation values
    print(f"Correlation between gradient norm and numerical derivative:")
    print(f"  Bayesian: {results['bayes_corr']:.4f}")
    print(f"  Deterministic: {results['det_corr']:.4f}")

# Plot the detailed analysis for the results we just computed
plot_detailed_gradient_analysis(results, start_class=2, target_class=9)


### Compare with Ensemble

In [ ]:
results = analyze_probability_gradients(
    bayes_model=bll_vi, 
    det_model=backbone, 
    centroids=centroids, 
    start_class=3, 
    target_class=6, 
    n_points=200,
    num_samples=200,
    ensemble_model=ensemble_model
)

In [ ]:
results = analyze_class_to_class_path(
    bayes_model=bll_vi, 
    det_model=backbone, 
    centroids=centroids, 
    start_class=3, 
    target_class=6, 
    n_points=200,
    num_samples=200,
    ensemble_model=ensemble_model)